# Final paper runs

Everything that is still to run for the paper, in one queue: the zero-shot baseline, timing and invalid-output checks on the finished adapters, archived Messidor scoring, and the new training variants (learning rate and alpha), each followed by its own scoring.

1. Run **1. Setup**, then **2. What is left**. Both only read files.
2. Set `TRAINING_MODE = True` in Setup and rerun it when you want to start GPU jobs.
3. Run **4. Pilot** first. It takes about 30 minutes and exercises every kind of job on a few images.
4. Run **5. Full queue** and leave it. It works through every unfinished job in order, keeps the PC awake, records a failed job and carries on. Rerunning the cell skips finished jobs and resumes interrupted training from its last epoch.

New results are written to `D:\Afterimage_Experiments\final_runs\<tag>`, apart from the 24 verified runs. Close other GPU-heavy apps before starting: the BF16 jobs need about 14 GB of free VRAM.

[Design and reasoning](../../../docs/paper/FINAL_RUNS_DESIGN.md) · [Deliverables checklist](../../../docs/paper/DELIVERABLES_CHECKLIST.md) · [Plan](../../../docs/paper/PLAN.md)

## 1. Setup

In [16]:
from pathlib import Path
import importlib
import sys

if Path(sys.prefix).name.casefold() != "afterimage_env" or sys.version_info[:2] != (3, 11):
    raise RuntimeError(f"Select the Python 3.11 afterimage_env kernel. Current: {sys.executable}")

sys.dont_write_bytecode = True
QWEN_DIR = Path(r"D:\Afterimage_Reproduction\finetuning\Qwen3")
HELPER_DIR = QWEN_DIR / "paper_revision"  # pinned paper_reproduction.py stays in QWEN_DIR
for directory in (QWEN_DIR, HELPER_DIR):
    if str(directory) in sys.path:
        sys.path.remove(str(directory))
    sys.path.insert(0, str(directory))
importlib.invalidate_caches()

def load_helper(name):
    module = importlib.reload(sys.modules[name]) if name in sys.modules else importlib.import_module(name)
    if Path(module.__file__).resolve() != (HELPER_DIR / f"{name}.py").resolve():
        raise ImportError(f"Expected {name} from {HELPER_DIR}, got {module.__file__}")
    return module

reproduction = load_helper("reproduce_run_02")
remaining = load_helper("remaining_runs")
final = load_helper("final_runs")

TRAINING_MODE = True  # Change to True and rerun this cell to enable GPU jobs.

SETTINGS = final.default_settings()
SETTINGS["tag"] = "final_01"        # one folder per campaign; keep it to resume or reuse results
SETTINGS["pilot_tag"] = "pilot_01"  # change it to repeat the pilot from scratch

# Stages. Turn one off to leave it out of the queue.
SETTINGS["stages"].update(
    zero_shot=True,               # S7: base model with no adapter
    benchmark=True,               # S14: time per image, and invalid outputs under text generation
    original_adapter_audit=True,  # invalid outputs of the original paper adapter (read from the C: copy)
    messidor_archived=True,       # archived 1,057-image Messidor cohort, existing adapters
    lr_study=True,                # G2/S8: learning rate, three seeds each
    alpha_study=True,             # S6: LoRA alpha, three seeds each
    extra_variants=True,          # anything listed in SETTINGS["extra_variants"]
    main_lr_grid=False,           # see section 8
    messidor_for_new_runs=True,   # score each new CLAHE-on adapter on archived Messidor
)
SETTINGS["lr_values"] = [1e-4, 2e-4]   # the finished runs used 5e-4
SETTINGS["alpha_values"] = [8, 32]     # the finished runs used 16
SETTINGS["study_rank"], SETTINGS["study_clahe"] = 16, "on"
SETTINGS["seed_pairs"] = [1, 2, 3]     # 42/3407, 43/3408, 44/3409

# Extra single runs, for example a dropout or patience trial:
# SETTINGS["extra_variants"] = [{"seed_pair": 1, "lora_dropout": 0.0},
#                               {"seed_pair": 1, "early_stopping_patience": 4}]

def launch(job, *args, **kwargs):
    if not TRAINING_MODE:
        print("Set TRAINING_MODE = True and rerun Setup, then run this cell.")
        return
    return job(*args, **kwargs)

print("Setup OK. Results folder:", final.root(SETTINGS))

Setup OK. Results folder: D:\Afterimage_Experiments\final_runs\final_01


## 2. What is left

Lists every job in the queue with its status and a time estimate. Reads saved files only.

In [17]:
jobs = final.show_plan(SETTINGS)

Final queue in D:\Afterimage_Experiments\final_runs\final_01
  [pending    ] zero_shot              zeroshot_4bit__aptos361_on  (~8 min)
  [pending    ] zero_shot              zeroshot_4bit__aptos361_off  (~8 min)
  [pending    ] zero_shot              zeroshot_bf16__aptos361_on  (~14 min)
  [pending    ] benchmark              r16_on_p1__aptos361_on  (~8 min)
  [pending    ] benchmark              r16_on_p2__aptos361_on  (~8 min)
  [pending    ] benchmark              r16_on_p3__aptos361_on  (~8 min)
  [pending    ] benchmark              r8_on_p3__aptos361_on  (~8 min)
  [pending    ] benchmark              r32_on_p1__aptos361_on  (~8 min)
  [pending    ] benchmark              bf16_r16_on_p1__aptos361_on  (~14 min)
  [pending    ] original_adapter_audit original_paper_adapter__aptos367_original  (~8 min)
  [pending    ] messidor_archived      zeroshot_4bit__messidor_archived  (~18 min)
  [pending    ] messidor_archived      r16_on_p1__messidor_archived  (~18 min)
  [pending    ] mes

## 3. Parameter counts (no GPU)

Counts the base model's parameters and each LoRA rank's trainable parameters directly from the weight files. The training logs disagree on the total (8.82 billion for 4-bit runs, 6.12 billion for BF16 runs), so this is the count to report.

In [18]:
counts = final.parameter_counts(SETTINGS)

Base model: 8,767,123,696 parameters.
  LoRA rank_8: 25,673,472 trainable (0.293% of the base), 98 MiB on disk.
  LoRA rank_16: 51,346,944 trainable (0.586% of the base), 196 MiB on disk.
  LoRA rank_32: 102,693,888 trainable (1.171% of the base), 392 MiB on disk.


## 4. Pilot

One small job of each kind: two 10-step training runs (a learning-rate variant and an alpha variant) and 12-image scoring runs for the zero-shot base, a finished 4-bit adapter, a finished BF16 adapter and archived Messidor. Check that the output ends with **Pilot passed** before starting the full queue.

In [19]:
pilot_jobs = final.show_plan(SETTINGS, pilot=True)

Pilot queue in D:\Afterimage_Experiments\final_runs\final_01__pilot_01
  [pending    ] zero_shot              zeroshot_4bit__aptos361_on  (~4 min)
  [done       ] benchmark              r16_on_p1__aptos361_on  (~4 min)
  [done       ] benchmark              bf16_r16_on_p1__aptos361_on  (~4 min)
  [pending    ] messidor_archived      zeroshot_4bit__messidor_archived  (~4 min)
  [done       ] lr_study               r16_on_lr0p0001_t42_a3407  (~6 min)
  [done       ] alpha_study            r16_on_alpha8_t42_a3407  (~6 min)
4 of 6 jobs done. Estimated GPU time left: about 0.1 h. Training times are estimates from earlier runs.


In [20]:
launch(final.run_queue, SETTINGS, pilot=True)


[1/6] zero_shot: zeroshot_4bit__aptos361_on
Log: D:\Afterimage_Experiments\final_runs\final_01__pilot_01\scores\zeroshot_4bit__aptos361_on\attempts\score-967ndp0q\score.log
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
W1115 21:23:50.565000 46344 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1115 21:23:50.638000 46344 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xfo

{'zeroshot_4bit__aptos361_on': 'passed',
 'r16_on_p1__aptos361_on': 'done earlier',
 'bf16_r16_on_p1__aptos361_on': 'done earlier',
 'zeroshot_4bit__messidor_archived': 'passed',
 'r16_on_lr0p0001_t42_a3407': 'done earlier',
 'r16_on_alpha8_t42_a3407': 'done earlier'}

## 5. Full queue

Runs every unfinished job in order and writes the summary at the end. Leave it running. If it is interrupted, run this cell again: finished jobs are skipped and training resumes from its last saved epoch.

In [21]:
launch(final.run_queue, SETTINGS)


[1/41] zero_shot: zeroshot_4bit__aptos361_on
Log: D:\Afterimage_Experiments\final_runs\final_01\scores\zeroshot_4bit__aptos361_on\attempts\score-mtq7jl2o\score.log
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
W1115 21:30:07.641000 46428 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1115 21:30:07.716000 46428 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, ru

{'zeroshot_4bit__aptos361_on': 'passed',
 'zeroshot_4bit__aptos361_off': 'passed',
 'zeroshot_bf16__aptos361_on': 'passed',
 'r16_on_p1__aptos361_on': 'passed',
 'r16_on_p2__aptos361_on': 'passed',
 'r16_on_p3__aptos361_on': 'passed',
 'r8_on_p3__aptos361_on': 'passed',
 'r32_on_p1__aptos361_on': 'passed',
 'bf16_r16_on_p1__aptos361_on': 'passed',
 'original_paper_adapter__aptos367_original': 'passed',
 'zeroshot_4bit__messidor_archived': 'passed',
 'r16_on_p1__messidor_archived': 'passed',
 'r16_on_p2__messidor_archived': 'passed',
 'r16_on_p3__messidor_archived': 'passed',
 'bf16_r16_on_p1__messidor_archived': 'passed',
 'bf16_r16_on_p2__messidor_archived': 'passed',
 'bf16_r16_on_p3__messidor_archived': 'passed',
 'r16_on_lr0p0001_t42_a3407': 'passed',
 'r16_on_lr0p0001_t42_a3407__messidor_archived': 'passed',
 'r16_on_lr0p0001_t43_a3408': 'passed',
 'r16_on_lr0p0001_t43_a3408__messidor_archived': 'passed',
 'r16_on_lr0p0001_t44_a3409': 'passed',
 'r16_on_lr0p0001_t44_a3409__messido

## 6. One stage at a time

Use these instead of the full queue if you want to run or repeat a single stage.

In [23]:
launch(final.run_queue, SETTINGS, only={"zero_shot"})


Queue finished.
  done earlier                 zeroshot_4bit__aptos361_on
  done earlier                 zeroshot_4bit__aptos361_off
  done earlier                 zeroshot_bf16__aptos361_on
Summary of 39 finished job(s): D:\Afterimage_Experiments\final_runs\final_01\summary.md


{'zeroshot_4bit__aptos361_on': 'done earlier',
 'zeroshot_4bit__aptos361_off': 'done earlier',
 'zeroshot_bf16__aptos361_on': 'done earlier'}

In [24]:
launch(final.run_queue, SETTINGS, only={"benchmark"})


Queue finished.
  done earlier                 r16_on_p1__aptos361_on
  done earlier                 r16_on_p2__aptos361_on
  done earlier                 r16_on_p3__aptos361_on
  done earlier                 r8_on_p3__aptos361_on
  done earlier                 r32_on_p1__aptos361_on
  done earlier                 bf16_r16_on_p1__aptos361_on
Summary of 39 finished job(s): D:\Afterimage_Experiments\final_runs\final_01\summary.md


{'r16_on_p1__aptos361_on': 'done earlier',
 'r16_on_p2__aptos361_on': 'done earlier',
 'r16_on_p3__aptos361_on': 'done earlier',
 'r8_on_p3__aptos361_on': 'done earlier',
 'r32_on_p1__aptos361_on': 'done earlier',
 'bf16_r16_on_p1__aptos361_on': 'done earlier'}

In [25]:
launch(final.run_queue, SETTINGS, only={"original_adapter_audit"})


Queue finished.
  done earlier                 original_paper_adapter__aptos367_original
Summary of 39 finished job(s): D:\Afterimage_Experiments\final_runs\final_01\summary.md


{'original_paper_adapter__aptos367_original': 'done earlier'}

In [26]:
launch(final.run_queue, SETTINGS, only={"messidor_archived"})


Queue finished.
  done earlier                 zeroshot_4bit__messidor_archived
  done earlier                 r16_on_p1__messidor_archived
  done earlier                 r16_on_p2__messidor_archived
  done earlier                 r16_on_p3__messidor_archived
  done earlier                 bf16_r16_on_p1__messidor_archived
  done earlier                 bf16_r16_on_p2__messidor_archived
  done earlier                 bf16_r16_on_p3__messidor_archived
Summary of 39 finished job(s): D:\Afterimage_Experiments\final_runs\final_01\summary.md


{'zeroshot_4bit__messidor_archived': 'done earlier',
 'r16_on_p1__messidor_archived': 'done earlier',
 'r16_on_p2__messidor_archived': 'done earlier',
 'r16_on_p3__messidor_archived': 'done earlier',
 'bf16_r16_on_p1__messidor_archived': 'done earlier',
 'bf16_r16_on_p2__messidor_archived': 'done earlier',
 'bf16_r16_on_p3__messidor_archived': 'done earlier'}

In [27]:
launch(final.run_queue, SETTINGS, only={"lr_study"})


Queue finished.
  done earlier                 r16_on_lr0p0001_t42_a3407
  done earlier                 r16_on_lr0p0001_t42_a3407__messidor_archived
  done earlier                 r16_on_lr0p0001_t43_a3408
  done earlier                 r16_on_lr0p0001_t43_a3408__messidor_archived
  done earlier                 r16_on_lr0p0001_t44_a3409
  done earlier                 r16_on_lr0p0001_t44_a3409__messidor_archived
  done earlier                 r16_on_lr0p0002_t42_a3407
  done earlier                 r16_on_lr0p0002_t42_a3407__messidor_archived
  done earlier                 r16_on_lr0p0002_t43_a3408
  done earlier                 r16_on_lr0p0002_t43_a3408__messidor_archived
  done earlier                 r16_on_lr0p0002_t44_a3409
  done earlier                 r16_on_lr0p0002_t44_a3409__messidor_archived
Summary of 39 finished job(s): D:\Afterimage_Experiments\final_runs\final_01\summary.md


{'r16_on_lr0p0001_t42_a3407': 'done earlier',
 'r16_on_lr0p0001_t42_a3407__messidor_archived': 'done earlier',
 'r16_on_lr0p0001_t43_a3408': 'done earlier',
 'r16_on_lr0p0001_t43_a3408__messidor_archived': 'done earlier',
 'r16_on_lr0p0001_t44_a3409': 'done earlier',
 'r16_on_lr0p0001_t44_a3409__messidor_archived': 'done earlier',
 'r16_on_lr0p0002_t42_a3407': 'done earlier',
 'r16_on_lr0p0002_t42_a3407__messidor_archived': 'done earlier',
 'r16_on_lr0p0002_t43_a3408': 'done earlier',
 'r16_on_lr0p0002_t43_a3408__messidor_archived': 'done earlier',
 'r16_on_lr0p0002_t44_a3409': 'done earlier',
 'r16_on_lr0p0002_t44_a3409__messidor_archived': 'done earlier'}

In [28]:
launch(final.run_queue, SETTINGS, only={"alpha_study"})


[9/12] alpha_study: r16_on_alpha32_t43_a3408
Log: D:\Afterimage_Experiments\final_runs\final_01\logs\r16_on_alpha32_t43_a3408\launch.log


Training is saved. Starting scoring in a new attempt.
Starting score. Log: D:\Afterimage_Experiments\final_runs\final_01\train\r16_on_alpha32_t43_a3408\attempts\20261116T165123Z-xm0myj3k\score.log
Report: D:\Afterimage_Experiments\final_runs\final_01\train\r16_on_alpha32_t43_a3408\attempts\20261116T165123Z-xm0myj3k\run_report.md
Traceback (most recent call last):
  File "D:\Afterimage_Reproduction\finetuning\Qwen3\paper_revision\qwen_variant.py", line 162, in <module>
    main()
  File "D:\Afterimage_Reproduction\finetuning\Qwen3\paper_revision\qwen_variant.py", line 137, in main
    recover(output, config, worker, {**vars(runner), "run_logged": logged})
  File "D:\Afterimage_Experiments\code\recover_qwen.py", line 49, in recover
    runner["run_logged"](output, config, ("score",))
  File "D:\Afterimage_Experiments\code\qwen_pilot.py", line 526, in run_logged
    raise RuntimeError(f"{stage} exited with code {code}. See {attempt / (stage + '.log')}")
RuntimeError: score exited with cod

{'r16_on_alpha8_t42_a3407': 'done earlier',
 'r16_on_alpha8_t42_a3407__messidor_archived': 'done earlier',
 'r16_on_alpha8_t43_a3408': 'done earlier',
 'r16_on_alpha8_t43_a3408__messidor_archived': 'done earlier',
 'r16_on_alpha8_t44_a3409': 'done earlier',
 'r16_on_alpha8_t44_a3409__messidor_archived': 'done earlier',
 'r16_on_alpha32_t42_a3407': 'done earlier',
 'r16_on_alpha32_t42_a3407__messidor_archived': 'done earlier',
 'r16_on_alpha32_t43_a3408': 'failed: Worker exited with code 1. See D:\\Afterimage_Experiments\\final_runs\\final_01\\logs\\r16_on_alpha32_t43_a3408\\launch.log.',
 'r16_on_alpha32_t43_a3408__messidor_archived': 'skipped: needs D:\\Afterimage_Experiments\\final_runs\\final_01\\train\\r16_on_alpha32_t43_a3408\\score_result.json',
 'r16_on_alpha32_t44_a3409': 'done earlier',
 'r16_on_alpha32_t44_a3409__messidor_archived': 'done earlier'}

In [29]:
launch(final.run_queue, SETTINGS, only={"extra_variants"})


Queue finished.
Summary of 39 finished job(s): D:\Afterimage_Experiments\final_runs\final_01\summary.md


{}

## 7. Results

Rebuilds `summary.csv` and `summary.md` from every finished job and shows the summary. Reads saved files only.

In [30]:
rows = final.summarize(SETTINGS)
summary = final.root(SETTINGS) / "summary.md"
if summary.is_file():
    from IPython.display import Markdown, display
    display(Markdown(summary.read_text(encoding="utf-8")))

Summary of 39 finished job(s): D:\Afterimage_Experiments\final_runs\final_01\summary.md


# Final runs summary (final_01)

Accuracy, QWK and macro AUC come from the pinned five-answer scorer. Times exclude model loading and the first three images.

| Stage | Job | Cohort | Images | Accuracy | QWK | Macro AUC | Median s/image | Unparseable generations |
|---|---|---|---|---|---|---|---|---|
| zero_shot | zeroshot_4bit__aptos361_on | aptos361_on | 361 | 46.54% | 0.6743 | 0.7725 | 0.712 | 0 of 361 |
| zero_shot | zeroshot_4bit__aptos361_off | aptos361_off | 361 | 54.57% | 0.7353 | 0.8034 | 0.718 | 0 of 361 |
| zero_shot | zeroshot_bf16__aptos361_on | aptos361_on | 361 | 52.63% | 0.6842 | 0.7658 | 1.344 | 0 of 361 |
| benchmark | r16_on_p1__aptos361_on | aptos361_on | 361 | 54.57% | 0.2807 | 0.8493 | 0.875 | 0 of 361 |
| benchmark | r16_on_p2__aptos361_on | aptos361_on | 361 | 80.33% | 0.8722 | 0.9243 | 0.852 | 0 of 361 |
| benchmark | r16_on_p3__aptos361_on | aptos361_on | 361 | 74.24% | 0.8573 | 0.9218 | 0.856 | 0 of 361 |
| benchmark | r8_on_p3__aptos361_on | aptos361_on | 361 | 75.07% | 0.9090 | 0.9349 | 0.896 | 0 of 361 |
| benchmark | r32_on_p1__aptos361_on | aptos361_on | 361 | 72.85% | 0.8547 | 0.9341 | 0.901 | 0 of 361 |
| benchmark | bf16_r16_on_p1__aptos361_on | aptos361_on | 361 | 75.90% | 0.8596 | 0.9331 | 1.348 | 0 of 361 |
| original_adapter_audit | original_paper_adapter__aptos367_original | aptos367_original | 367 | 81.47% | 0.9286 | 0.9488 | 0.845 | 0 of 367 |
| messidor_archived | zeroshot_4bit__messidor_archived | messidor_archived | 1057 | 45.98% | 0.4536 | 0.7349 | 0.667 |  |
| messidor_archived | r16_on_p1__messidor_archived | messidor_archived | 1057 | 43.52% | 0.1829 | 0.7380 | 0.782 |  |
| messidor_archived | r16_on_p2__messidor_archived | messidor_archived | 1057 | 48.16% | 0.5090 | 0.7503 | 0.785 |  |
| messidor_archived | r16_on_p3__messidor_archived | messidor_archived | 1057 | 51.66% | 0.6285 | 0.7837 | 0.781 |  |
| messidor_archived | bf16_r16_on_p1__messidor_archived | messidor_archived | 1057 | 45.70% | 0.5401 | 0.7792 | 1.335 |  |
| messidor_archived | bf16_r16_on_p2__messidor_archived | messidor_archived | 1057 | 49.67% | 0.6214 | 0.7935 | 1.334 |  |
| messidor_archived | bf16_r16_on_p3__messidor_archived | messidor_archived | 1057 | 54.12% | 0.6448 | 0.7964 | 1.333 |  |
| lr_study | r16_on_lr0p0001_t42_a3407 | aptos361_on | 361 | 75.07% | 0.9062 | 0.9357 |  |  |
| lr_study | r16_on_lr0p0001_t42_a3407__messidor_archived | messidor_archived | 1057 | 52.03% | 0.6105 | 0.8032 | 0.819 |  |
| lr_study | r16_on_lr0p0001_t43_a3408 | aptos361_on | 361 | 72.85% | 0.8483 | 0.9234 |  |  |
| lr_study | r16_on_lr0p0001_t43_a3408__messidor_archived | messidor_archived | 1057 | 45.70% | 0.4974 | 0.7755 | 0.817 |  |
| lr_study | r16_on_lr0p0001_t44_a3409 | aptos361_on | 361 | 77.01% | 0.8841 | 0.9354 |  |  |
| lr_study | r16_on_lr0p0001_t44_a3409__messidor_archived | messidor_archived | 1057 | 48.06% | 0.5554 | 0.8016 | 0.820 |  |
| lr_study | r16_on_lr0p0002_t42_a3407 | aptos361_on | 361 | 75.90% | 0.8890 | 0.9436 |  |  |
| lr_study | r16_on_lr0p0002_t42_a3407__messidor_archived | messidor_archived | 1057 | 49.57% | 0.6277 | 0.8113 | 0.816 |  |
| lr_study | r16_on_lr0p0002_t43_a3408 | aptos361_on | 361 | 80.33% | 0.9018 | 0.9439 |  |  |
| lr_study | r16_on_lr0p0002_t43_a3408__messidor_archived | messidor_archived | 1057 | 49.95% | 0.6055 | 0.7912 | 0.919 |  |
| lr_study | r16_on_lr0p0002_t44_a3409 | aptos361_on | 361 | 78.12% | 0.8909 | 0.9370 |  |  |
| lr_study | r16_on_lr0p0002_t44_a3409__messidor_archived | messidor_archived | 1057 | 46.07% | 0.5754 | 0.7943 | 0.917 |  |
| alpha_study | r16_on_alpha8_t42_a3407 | aptos361_on | 361 | 74.52% | 0.8903 | 0.9455 |  |  |
| alpha_study | r16_on_alpha8_t42_a3407__messidor_archived | messidor_archived | 1057 | 48.34% | 0.6290 | 0.8075 | 0.913 |  |
| alpha_study | r16_on_alpha8_t43_a3408 | aptos361_on | 361 | 72.85% | 0.8720 | 0.9301 |  |  |
| alpha_study | r16_on_alpha8_t43_a3408__messidor_archived | messidor_archived | 1057 | 48.91% | 0.6475 | 0.8131 | 0.853 |  |
| alpha_study | r16_on_alpha8_t44_a3409 | aptos361_on | 361 | 8.31% | 0.0000 | 0.5419 |  |  |
| alpha_study | r16_on_alpha8_t44_a3409__messidor_archived | messidor_archived | 1057 | 1.99% | 0.0000 | 0.4802 | 0.846 |  |
| alpha_study | r16_on_alpha32_t42_a3407 | aptos361_on | 361 | 50.97% | 0.3596 | 0.7698 |  |  |
| alpha_study | r16_on_alpha32_t42_a3407__messidor_archived | messidor_archived | 1057 | 44.28% | 0.0000 | 0.4599 | 0.890 |  |
| alpha_study | r16_on_alpha32_t44_a3409 | aptos361_on | 361 | 8.31% | 0.0000 | 0.3969 |  |  |
| alpha_study | r16_on_alpha32_t44_a3409__messidor_archived | messidor_archived | 1057 | 1.99% | 0.0000 | 0.4694 | 0.839 |  |

## Training variants, mean over seeds

| Stage | Rank | Cohort | lr | alpha | dropout | patience | Seeds | Accuracy mean ± SD | QWK mean | Macro AUC mean | Constant predictors |
|---|---|---|---|---|---|---|---|---|---|---|---|
| lr_study | 16 | aptos361_on | 0.0001 | 16 | 0.05 | 2 | 3 | 74.98 ± 2.08 | 0.8795 | 0.9315 | 0 |
| lr_study | 16 | aptos361_on | 0.0002 | 16 | 0.05 | 2 | 3 | 78.12 ± 2.22 | 0.8939 | 0.9415 | 0 |
| alpha_study | 16 | aptos361_on | 0.0005 | 8 | 0.05 | 2 | 3 | 51.89 ± 37.75 | 0.5874 | 0.8058 | 1 |
| alpha_study | 16 | aptos361_on | 0.0005 | 32 | 0.05 | 2 | 2 | 29.64 ± 30.16 | 0.1798 | 0.5833 | 1 |


## 8. After the learning-rate study: repeat the grid at the chosen rate

Only if you and Dr. Noor decide to adopt a different learning rate. Choose it from the **validation** losses in the summary, not from test accuracy. This repeats rank 8/16/32 with CLAHE on and off at that rate, three seeds each, skipping runs the study already produced. Expect about 20 hours.

In [15]:
# SETTINGS["main_lr"] = 1e-4
# SETTINGS["stages"]["main_lr_grid"] = True
# final.show_plan(SETTINGS)
# launch(final.run_queue, SETTINGS, only={"main_lr_grid"})

## 9. Not covered here

- **BF16 variants** (CLAHE off, other learning rates): the BF16 worker only accepts the rank 16, CLAHE-on reference. A new worker is needed.
- **RetiZero**: its own notebook, `finetuning/RetiZero/notebooks/02_retizero_runs.ipynb`.
- **Verified Messidor**: section 8 of `021_training.ipynb`, once the raw data is located. The Messidor numbers here use the archived 1,057-image export.
- **Paper tables**: these runs are summarised here. They are not yet read by `finetuning/paper_tables/build_paper_tables.py`.